In [ ]:
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"



In [ ]:
### Quantidade de municípios
from ibge.models import Municipio

Municipio.objects.count()

In [ ]:
### Listar municípios
Municipio.objects.all()[:10]

In [ ]:
Municipio.objects.get(nome="Mauá")

In [ ]:
from ibge.models import Indicador

Indicador.objects.all().values("id", "codigo", "nome")

In [ ]:
## Ranking de população

from ibge.models import FatoIndicador, Indicador, Tempo

# pegar o último ano disponível para população
pop = Indicador.objects.get(codigo="POPULACAO")

ultimo_ano = (
    FatoIndicador.objects.filter(indicador=pop)
    .order_by("-tempo__ano")
    .values_list("tempo_id", flat=True)
    .first()
)

ranking = (
    FatoIndicador.objects.filter(indicador=pop, tempo=ultimo_ano)
    .select_related("municipio", "municipio__estado", "tempo")
    .order_by("-valor")[:10]
)

for posicao, item in enumerate(ranking, start=1):
    print(
        f"{posicao}º - "
        f"{item.municipio.nome}/{item.municipio.estado.sigla}: "
        f"{int(item.valor):,}".replace(",", "."),
        f"({item.tempo.ano})",
    )

In [91]:
from ibge.models import FatoIndicador, Indicador, Tempo

# pegar o último ano disponível para pib
pib = Indicador.objects.get(codigo="PIB")

ultimo_ano = (
    FatoIndicador.objects.filter(indicador=pib)
    .order_by("-tempo__ano")
    .values_list("tempo_id", flat=True)
    .first()
)

## Top 10 PIB
ranking = (
    FatoIndicador.objects.filter(indicador__codigo="PIB", tempo=ultimo_ano)
    .select_related("municipio")
    .order_by("-valor")[:10]
)

for posicao, item in enumerate(ranking, start=1):
    valor = float(item.valor)

    valor_formatado = (
        f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    print(f"{posicao}ª {item.municipio.nome}: {valor_formatado}")

1ª São Paulo: R$ 945.946.483,00
2ª Rio de Janeiro: R$ 380.381.515,00
3ª Brasília: R$ 328.789.563,00
4ª Maricá: R$ 158.394.291,00
5ª Belo Horizonte: R$ 115.381.698,00
6ª Manaus: R$ 113.465.435,00
7ª Osasco: R$ 112.145.798,00
8ª Curitiba: R$ 111.760.270,00
9ª Porto Alegre: R$ 90.214.952,00
10ª Niterói: R$ 88.716.289,00
